In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random
import ast
import plotly.graph_objects as go

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

In [ ]:
sensor_df = pd.read_parquet("sensor.parquet")
event_df = pd.read_parquet("event.parquet")

print(sensor_df.shape, event_df.shape)

In [ ]:
def safe_parse_array(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return np.array(x, dtype=np.float32)

    if isinstance(x, str):
        try:
            return np.array(ast.literal_eval(x), dtype=np.float32)
        except:
            x = x.replace("[", "").replace("]", "")
            return np.fromstring(x, sep=" ", dtype=np.float32)

    return np.zeros(0, dtype=np.float32)

In [ ]:
LAT_MEAN = sensor_df["latitude"].mean()
LON_MEAN = sensor_df["longitude"].mean()

LAT_STD = sensor_df["latitude"].std() + 1e-6
LON_STD = sensor_df["longitude"].std() + 1e-6

def normalize_xy(xy):
    xy[:, 0] = (xy[:, 0] - LAT_MEAN) / LAT_STD
    xy[:, 1] = (xy[:, 1] - LON_MEAN) / LON_STD
    return xy

In [ ]:
def build_subtracks(sensor_df, min_len=5, max_len=10, stride=2):
    sub_tracks = []

    for tid, g in sensor_df.groupby("trackId"):
        g = g.sort_values("timestamp")
        xy = g[["latitude", "longitude"]].values

        for w in range(min_len, max_len + 1):
            for i in range(0, len(xy) - w + 1, stride):
                seg = xy[i:i+w].copy()
                seg = normalize_xy(seg)

                sub_tracks.append({
                    "track_id": tid,
                    "segment": seg
                })

    return sub_tracks

sub_tracks = build_subtracks(sensor_df)
print("Subtracks:", len(sub_tracks))

In [ ]:
def encode_report_seq(row, L=10):
    lons = safe_parse_array(row["coors.longitudes"])
    lats = safe_parse_array(row["coors.latitudes"])

    n = min(len(lons), len(lats))
    if n == 0:
        return np.zeros((L, 2), dtype=np.float32)

    xy = np.stack([lats[:n], lons[:n]], axis=1)

    # CASE 1: only 1 point → replicate
    if len(xy) == 1:
        xy = np.repeat(xy, L, axis=0)

    # CASE 2: pad if short
    if len(xy) < L:
        pad = np.repeat(xy[-1:], L - len(xy), axis=0)
        xy = np.vstack([xy, pad])

    # CASE 3: truncate
    xy = xy[:L]

    return normalize_xy(xy)

In [ ]:
track_seqs = [s["segment"] for s in sub_tracks]
report_seqs = [encode_report_seq(r) for _, r in event_df.iterrows()]

In [ ]:
class TrajTransformer(nn.Module):
    def __init__(self, d_model=64):
        super().__init__()

        self.input_proj = nn.Linear(2, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.encoder(x)
        return x.mean(dim=1)

In [ ]:
class PairDataset(Dataset):
    def __init__(self, track_seqs, report_seqs, num_neg=3):
        self.data = []

        for rid, r in enumerate(report_seqs):
            pos = random.randint(0, len(track_seqs)-1)

            for _ in range(num_neg):
                neg = random.randint(0, len(track_seqs)-1)

                self.data.append((r, track_seqs[pos], track_seqs[neg]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        r, p, n = self.data[idx]
        return (
            torch.tensor(r),
            torch.tensor(p),
            torch.tensor(n)
        )

In [ ]:
def spatial_loss(anchor, pos, neg, pos_xy, neg_xy, w_spatial=0.5):
    d_pos = F.pairwise_distance(anchor, pos)
    d_neg = F.pairwise_distance(anchor, neg)

    loss_embed = F.relu(d_pos - d_neg + 1.0)

    spatial_pos = ((pos_xy - pos_xy.mean(dim=1, keepdim=True))**2).sum()
    spatial_neg = ((neg_xy - neg_xy.mean(dim=1, keepdim=True))**2).sum()

    loss_spatial = spatial_pos - spatial_neg

    return (loss_embed + w_spatial * loss_spatial).mean()

In [ ]:
net = TrajTransformer()

dataset = PairDataset(track_seqs, report_seqs)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

for ep in range(10):
    total = 0

    for r, p, n in loader:
        optimizer.zero_grad()

        e_r = net(r.float())
        e_p = net(p.float())
        e_n = net(n.float())

        loss = spatial_loss(e_r, e_p, e_n, p, n)
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {ep+1}: {total/len(loader):.4f}")

In [ ]:
def match_report_topk(net, report_seqs, track_seqs, sub_tracks, top_k=5):
    net.eval()

    with torch.no_grad():
        E_t = net(torch.tensor(track_seqs).float())
        E_r = net(torch.tensor(report_seqs).float())

        dist = torch.cdist(E_r, E_t)  # 🔥 report → track

    results = {}

    for rid in range(len(report_seqs)):
        row = dist[rid].cpu().numpy()
        idxs = np.argsort(row)[:top_k]

        results[rid] = [
            {
                "track_id": sub_tracks[i]["track_id"],
                "score": float(row[i])
            }
            for i in idxs
        ]

    return results

In [ ]:
def plot_report_vs_tracks(sensor_df, event_df, topk_results, rid=0):
    fig = go.Figure()

    # REPORT
    row = event_df.iloc[rid]
    lons = safe_parse_array(row["coors.longitudes"])
    lats = safe_parse_array(row["coors.latitudes"])

    fig.add_trace(go.Scatter(
        x=lons,
        y=lats,
        mode="lines+markers",
        name="Report",
        line=dict(width=4)
    ))

    # TRACKS
    for item in topk_results[rid]:
        tid = item["track_id"]

        df = sensor_df[sensor_df["trackId"] == tid].sort_values("timestamp")

        fig.add_trace(go.Scatter(
            x=df["longitude"],
            y=df["latitude"],
            mode="lines",
            name=f"Track {tid}"
        ))

    fig.update_layout(
        title=f"Report {rid} vs Top Tracks",
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        height=600
    )

    return fig

In [ ]:
fig = plot_report_vs_tracks(sensor_df, event_df, topk_results, rid=0)
fig.show()